In [1]:
"""
============================================================
AILA UNIFIED PIPELINE  —  A100 GPU
Preprocessing → Hard Negative Mining → Training → Evaluation
============================================================

Directory layout expected:
  BASE/
    Object_casedocs/       ← raw case .txt files
    Object_statutes/       ← raw statute .txt files
    Query_doc.txt          ← queries (qid || text per line)
    relevance_judgments_priorcases.txt
    relevance_judgments_statutes.txt

Outputs written under BASE/:
  processed_cases/         Stage 1 JSON
  enriched_cases/          Stage 2 JSON
  refined_cases/           Stage 3.5 JSON
  compact_data/            compact_queries/cases/statutes JSON
  training_data_v3/        BM25 triplets + pairs
  semantic_hard_v1/        semantic triplets
  models/bi_encoder_v1/
  models/bi_encoder_v2/
  models/bi_encoder_v3_a100/
"""

# ============================================================
# 0. INSTALL
# ============================================================
import subprocess, sys

def pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

pip("sentence-transformers", "accelerate", "faiss-cpu", "rank-bm25")


# ============================================================
# 1. IMPORTS & SEEDS
# ============================================================
import os, re, gc, json, glob, random
import numpy as np
import torch
from pathlib import Path
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, InputExample, losses
from rank_bm25 import BM25Okapi
import faiss

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ─── CHANGE THIS ────────────────────────────────────────────
BASE = "/workspace/aila/ailanew/aila"
# ────────────────────────────────────────────────────────────

CASE_DIR  = f"{BASE}/Object_casedocs"
STAT_DIR  = f"{BASE}/Object_statues"
QUERY_FILE = f"{BASE}/Query_doc.txt"
CASE_REL   = f"{BASE}/relevance_judgments_priorcases.txt"
STAT_REL   = f"{BASE}/relevance_judgments_statutes.txt"
# aila/ailanew/aila/Object_casedocs
for d in [
    f"{BASE}/processed_cases", f"{BASE}/enriched_cases",
    f"{BASE}/refined_cases",   f"{BASE}/compact_data",
    f"{BASE}/training_data_v3", f"{BASE}/semantic_hard_v1",
    f"{BASE}/models/bi_encoder_v1", f"{BASE}/models/bi_encoder_v2",
    f"{BASE}/models/bi_encoder_v3_a100",
]:
    os.makedirs(d, exist_ok=True)

print("GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
if torch.cuda.is_available():
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory/1e9, 2), "GB")


# ============================================================
# 2. SHARED HELPERS
# ============================================================
def load_json(fp):
    with open(fp, "r", encoding="utf-8") as f:
        return json.load(f)

def save_json(obj, fp, **kw):
    with open(fp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, **kw)

def parse_rel(path):
    """Return {qid: set(relevant_doc_ids)} for label==1."""
    gt = defaultdict(set)
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            p = line.strip().split()
            if len(p) == 4:
                qid, _, did, lab = p
                if int(lab) == 1:
                    gt[qid].add(did)
    return gt

STOP = {
    "the","and","for","with","that","this","shall","have","been","from",
    "into","which","such","their","there","where","when","what","whose",
    "within","under","against","every","person","other","also","than","then","they"
}


# ============================================================
# 3. CASE PREPROCESSING  (Stage 1 → 3.5)
# ============================================================

# ── 3a. Clean text ──────────────────────────────────────────
def clean_text(text):
    text = text.replace("\x0c", " ").replace("\t", " ")
    text = re.sub(r"\r", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ ]{2,}", " ", text)
    text = re.sub(r"[-_=*]{4,}", " ", text)
    text = re.sub(r"\n\s*\d+\s*\n", "\n", text)
    text = re.sub(r"\b[Ss](ection|ec\.?)\s*(\d+[A-Za-z\-]*)", r"SECTION_\2", text)
    text = re.sub(r"\b[Aa]rticle\s*(\d+[A-Za-z\-]*)", r"ARTICLE_\1", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# ── 3b. Weighted role classifier ────────────────────────────
ROLE_WEIGHTS = {
    "HEADER":  {"supreme court":2.5,"high court":2.5,"judgment was delivered":2.0,"bench":1.0},
    "FACT":    {"facts":2.0,"background":1.2,"agreement":1.0,"notice":0.8,"entered into":1.0,"filed":0.6},
    "ISSUE":   {"whether":1.2,"question":1.0,"issue":1.0,"controversy":1.2},
    "STATUTE": {"section_":2.0,"article_":2.0,"act":0.6,"constitution":1.5,"code":1.0,"rule":0.8},
    "PRECEDENT":{"v ":2.2," v. ":2.2,"versus":2.2,"scc":1.5,"air ":1.5,"held in":0.8},
    "ARGUMENT":{"submitted":1.2,"contended":1.4,"argued":1.2,"according to":0.8,"learned counsel":1.8},
    "PROCEDURE":{"petition":1.0,"appeal":1.0,"writ":1.2,"trial":1.2,"bench":0.8,"magistrate":0.8},
    "REASONING":{"therefore":2.0,"hence":1.6,"thus":1.4,"in our view":2.0,"we find":1.8,"considering":0.8},
    "HOLDING":  {"we hold":3.0,"we conclude":3.0,"held that":2.0,"find no merit":3.0,"must be accepted":2.5},
    "ORDER":    {"appeal dismissed":4.0,"appeal allowed":4.0,"petition dismissed":4.0,
                 "dismiss the appeal":4.0,"set aside":3.0,"disposed of":3.0},
}
ROLE_IMPORTANCE = {
    "HEADER":0.10,"FACT":0.55,"ISSUE":0.80,"STATUTE":0.75,"PRECEDENT":0.70,
    "ARGUMENT":0.50,"PROCEDURE":0.60,"REASONING":0.90,"HOLDING":0.98,"ORDER":1.00,"OTHER":0.30,
}

def classify_weighted(text):
    txt = text.lower()
    scores = {}
    for role, lex in ROLE_WEIGHTS.items():
        sc = sum(w for phrase, w in lex.items() if phrase in txt)
        if sc > 0:
            scores[role] = sc
    if not scores:
        return "OTHER", {"OTHER":1.0}, 0.30
    primary = max(scores, key=scores.get)
    mx = max(scores.values())
    norm = {k: round(v/mx, 3) for k, v in scores.items()}
    return primary, norm, ROLE_IMPORTANCE.get(primary, 0.30)

# ── 3c. Legal phrase / keyword helpers ──────────────────────
CASE_PAT = re.compile(r'\b[A-Z][A-Za-z.&,\- ]{1,80}\s+v\.?\s+[A-Z][A-Za-z.&,\- ]{1,80}\b')
SEC_PAT  = re.compile(
    r'\b(?:SECTION|ARTICLE)_[A-Za-z0-9\-]+|\bSection\s+\d+[A-Za-z\-]*'
    r'|s[\.\-]?\d+[A-Za-z\-]*|\bParagraph\s+\d+', re.I)
YEAR_PAT  = re.compile(r'\b(18|19|20)\d{2}\b')
MONEY_PAT = re.compile(r'Rs\.?\s?[\d,]+(?:\.\d+)?')
COURT_PAT = re.compile(r'(Supreme Court of India|High Court|Tribunal)', re.I)
ACT_PAT   = re.compile(r'\b(?:IPC|CrPC|CPC|Constitution|Act)\b', re.I)

def extract_legal_phrases(text, topk=12):
    toks = re.findall(r'\b[a-zA-Z]{3,}\b', text.lower())
    phrases = []
    keywords = {
        "act","court","issue","appeal","burden","criminal","foreigner",
        "liability","evidence","petition","cheque","prosecution","judge",
        "conviction","acquittal","right","liberty",
    }
    for n in [2, 3]:
        for i in range(len(toks)-n+1):
            p = " ".join(toks[i:i+n])
            if any(k in p for k in keywords):
                phrases.append(p)
    return [x for x, _ in Counter(phrases).most_common(topk)]

def extract_keywords(text, topk=25):
    words = [w for w in re.findall(r'\b[a-zA-Z]{4,}\b', text.lower()) if w not in STOP]
    return [w for w, _ in Counter(words).most_common(topk)]

def extract_cases(text):
    clean = []
    for x in CASE_PAT.findall(text):
        x = re.sub(r'\s+(Supreme Court|High Court|AIR).*', '', x).strip(" .,")
        if 8 < len(x) < 120:
            clean.append(x)
    return list(dict.fromkeys(clean))[:100]

def extract_metadata(text):
    return {
        "years": list(set("".join(y) for y in YEAR_PAT.findall(text)))[:20],
        "money_mentions": MONEY_PAT.findall(text)[:20],
        "courts": list(set(COURT_PAT.findall(text)))[:10],
        "section_count": len(SEC_PAT.findall(text)),
        "citation_count": len(CASE_PAT.findall(text)),
    }

# ── 3d. Overlapping chunker ─────────────────────────────────
def split_sentences(text):
    sents = re.split(r'(?<=[.!?])\s+(?=[A-Z])', text)
    return [s.strip() for s in sents if len(s.strip()) > 25]

def overlap_chunk(sentences, size=6, stride=3):
    chunks = []
    for i in range(0, len(sentences), stride):
        block = sentences[i:i+size]
        if len(block) < 2:
            continue
        chunks.append(" ".join(block))
        if i + size >= len(sentences):
            break
    return chunks

# ── 3e. Process one case file  (Stages 1 → 3.5 fused) ──────
def process_case_file(fp):
    doc_id = Path(fp).stem
    with open(fp, "r", encoding="utf-8", errors="ignore") as f:
        raw = f.read()

    clean = clean_text(raw)
    sentences = split_sentences(clean)
    chunks = overlap_chunk(sentences)

    passages = []
    for ch in chunks:
        role, role_scores, importance = classify_weighted(ch)
        passages.append({
            "primary_role": role,
            "role_scores": role_scores,
            "importance": importance,
            "keywords": extract_legal_phrases(ch),
            "text": ch,
        })

    full = clean
    out = {
        "doc_id": doc_id,
        "entities": {
            "acts": list(set(ACT_PAT.findall(full)))[:50],
            "sections": list(set(SEC_PAT.findall(full)))[:100],
            "cases_cited": extract_cases(full),
        },
        "metadata": extract_metadata(full),
        "global_keywords": extract_keywords(full),
        "n_passages": len(passages),
        "passages": passages,
    }
    save_json(out, f"{BASE}/refined_cases/{doc_id}.json", indent=2)
    return True

def safe_case(fp):
    try:
        process_case_file(fp)
        return True, None
    except Exception as e:
        return False, str(e)

print("\n[STAGE 3] Preprocessing cases...")
case_files = sorted(glob.glob(f"{CASE_DIR}/*.txt"))
print(f"  Found {len(case_files)} case files")
ok = 0; bad = []
with ThreadPoolExecutor(max_workers=min(16, os.cpu_count()*2)) as ex:
    for s, e in tqdm(
        (f.result() for f in [ex.submit(safe_case, fp) for fp in case_files]),
        total=len(case_files), desc="cases"
    ):
        if s: ok += 1
        else: bad.append(e)
print(f"  OK={ok}  Failed={len(bad)}")


# ============================================================
# 4. QUERY PREPROCESSING
# ============================================================
print("\n[STAGE 4] Preprocessing queries...")

def preprocess_query(qid, text):
    text = clean_text(text)
    sentences = split_sentences(text)
    chunks = overlap_chunk(sentences)
    passages = []
    for ch in chunks:
        role, _, _ = classify_weighted(ch)
        passages.append({"role": role, "keywords": extract_legal_phrases(ch), "text": ch})
    return {
        "qid": qid,
        "text": text,
        "entities": {
            "acts": list(set(ACT_PAT.findall(text)))[:50],
            "sections": list(set(SEC_PAT.findall(text)))[:100],
            "cases_cited": extract_cases(text),
        },
        "metadata": extract_metadata(text),
        "global_keywords": extract_legal_phrases(text),
        "n_passages": len(passages),
        "passages": passages,
    }

processed_queries = []
with open(QUERY_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if "||" not in line:
            continue
        qid, text = line.strip().split("||", 1)
        processed_queries.append(preprocess_query(qid.strip(), text.strip()))

save_json(processed_queries, f"{BASE}/processed_queries.json", indent=2)
print(f"  Queries: {len(processed_queries)}")


# ============================================================
# 5. STATUTE PREPROCESSING
# ============================================================
print("\n[STAGE 5] Preprocessing statutes...")

DOMAIN_WORDS = {
    "constitutional": ["article","constitution","right","liberty","high court","supreme court","writ"],
    "criminal": ["offence","crime","punishment","murder","theft","culpable","assembly","imprisonment"],
    "procedure": ["magistrate","investigation","police","charge","cognizance"],
    "labour": ["workman","industrial dispute","labour court","tribunal","employee","wages"],
    "property": ["land","acquisition","compensation","award","collector"],
    "arbitration": ["arbitration","award","arbitrator"],
    "civil": ["property","agreement","contract","damages","liability"],
}

def detect_act(text):
    t = text.lower()
    if "constitution" in t or "article" in t:
        return "Constitution of India"
    if any(x in t for x in ["indian penal code","offence","culpable homicide","theft","murder"]):
        return "Indian Penal Code"
    if any(x in t for x in ["magistrate","investigation","police officer","charge sheet"]):
        return "Code of Criminal Procedure"
    if any(x in t for x in ["labour court","industrial dispute","workman","tribunal","conciliation"]):
        return "Industrial Disputes Act"
    if any(x in t for x in ["land acquisition","collector","compensation awarded"]):
        return "Land Acquisition Act"
    if any(x in t for x in ["arbitrator","award","arbitration"]):
        return "Arbitration Act"
    return "General Law"

def detect_domain(text):
    t = text.lower()
    scores = {dom: sum(1 for w in words if w in t) for dom, words in DOMAIN_WORDS.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else "general"

def extract_section_ref(title, desc):
    txt = title + " " + desc
    for pat, fmt in [
        (r'\bArticle\s+(\d+[A-Z\-]*)\b', "Article {}"),
        (r'\bSection\s+(\d+[A-Z\-]*)\b', "Section {}"),
        (r'\bsub-?section\s*\(?(\d+[A-Z\-]*)\)?', "Subsection {}"),
    ]:
        m = re.search(pat, txt, re.I)
        if m:
            return fmt.format(m.group(1))
    return None

def chunk_text(text, size=120, stride=60):
    words = text.split()
    chunks = []
    for i in range(0, len(words), stride):
        block = words[i:i+size]
        if len(block) >= 25:
            chunks.append(" ".join(block))
        if i + size >= len(words):
            break
    return chunks

def parse_statute(fp):
    with open(fp, "r", encoding="utf-8") as f:
        raw = f.read()
    m1 = re.search(r'Title:\s*(.*)', raw)
    m2 = re.search(r'Desc:\s*(.*)', raw, re.S)
    title = m1.group(1).strip() if m1 else ""
    desc  = m2.group(1).strip() if m2 else ""
    desc = re.sub(r'\[[^\]]*\]', ' ', desc)
    desc = re.sub(r'\s+', ' ', desc).strip()
    full = f"{title}. {desc}"
    return {
        "sid": Path(fp).stem,
        "title": title,
        "act": detect_act(full),
        "section": extract_section_ref(title, desc),
        "domain": detect_domain(full),
        "keywords": extract_keywords(full),
        "concepts": extract_legal_phrases(full, topk=30),
        "chunks": chunk_text(full),
        "embedding_text": full,
    }

stat_files = sorted(glob.glob(f"{STAT_DIR}/*.txt"))
print(f"  Found {len(stat_files)} statute files")
processed_statutes = []
with ThreadPoolExecutor(max_workers=16) as ex:
    for fut in tqdm(as_completed([ex.submit(parse_statute, fp) for fp in stat_files]),
                    total=len(stat_files), desc="statutes"):
        try:
            processed_statutes.append(fut.result())
        except Exception:
            pass
save_json(processed_statutes, f"{BASE}/processed_statutes.json", indent=2)
print(f"  Statutes: {len(processed_statutes)}")


# ============================================================
# 6. BUILD COMPACT REPRESENTATIONS
# ============================================================
print("\n[STAGE 6] Building compact representations...")

IMPORTANT_ROLES = {"ISSUE","STATUTE","PRECEDENT","REASONING","HOLDING","ORDER"}
SECONDARY_ROLES  = {"FACT","ARGUMENT","PROCEDURE"}

def _join(items, n=15): return ", ".join(str(x).strip() for x in items[:n] if str(x).strip())

def compact_query(q):
    parts = []
    e = q.get("entities", {})
    if e.get("acts"):      parts.append("ACTS: "     + _join(e["acts"], 8))
    if e.get("sections"):  parts.append("SECTIONS: " + _join(e["sections"], 10))
    if e.get("cases_cited"): parts.append("CASES: "  + _join(e["cases_cited"], 8))
    if q.get("global_keywords"): parts.append("KEYWORDS: " + _join(q["global_keywords"], 20))
    role_blocks = [
        f"{p['role']}: {p['text'][:450]}"
        for p in q.get("passages", [])
        if p.get("role") in IMPORTANT_ROLES and len(p.get("text","")) >= 40
    ]
    if len(role_blocks) < 2:
        role_blocks += [
            f"{p['role']}: {p['text'][:350]}"
            for p in q.get("passages", [])
            if p.get("role") in SECONDARY_ROLES and len(p.get("text","")) >= 40
        ]
    parts.extend(role_blocks[:6])
    return "\n".join(parts)

def compact_case(obj):
    parts = []
    e = obj.get("entities", {})
    if e.get("acts"):      parts.append("ACTS: "     + _join(e["acts"], 8))
    if e.get("sections"):  parts.append("SECTIONS: " + _join(e["sections"], 10))
    if e.get("cases_cited"): parts.append("CASES: "  + _join(e["cases_cited"], 10))
    if obj.get("global_keywords"): parts.append("KEYWORDS: " + _join(obj["global_keywords"], 20))
    passages = sorted(obj.get("passages",[]), key=lambda x: x.get("importance",0), reverse=True)
    seen = set(); chosen = []
    for p in passages:
        role = p.get("primary_role","OTHER"); txt = p.get("text","").strip()
        if role == "HEADER" or len(txt) < 60: continue
        if role not in IMPORTANT_ROLES and role not in SECONDARY_ROLES: continue
        if role in seen and role in IMPORTANT_ROLES: continue
        chosen.append(f"{role}: {txt[:600]}")
        seen.add(role)
        if len(chosen) >= 6: break
    parts.extend(chosen)
    return "\n".join(parts)

def compact_statute(s):
    parts = []
    if s.get("act"):    parts.append(f"ACT: {s['act']}")
    if s.get("section"): parts.append(f"SECTION: {s['section']}")
    if s.get("title"):  parts.append(f"TITLE: {s['title']}")
    if s.get("domain"): parts.append(f"DOMAIN: {s['domain']}")
    if s.get("keywords"): parts.append("KEYWORDS: " + _join(s["keywords"], 20))
    if s.get("concepts"): parts.append("CONCEPTS: " + _join(s["concepts"], 20))
    if s.get("chunks"): parts.append("TEXT: " + s["chunks"][0][:900])
    return "\n".join(parts)

# queries
with open(f"{BASE}/processed_queries.json", "r", encoding="utf-8") as f:
    queries = json.load(f)
compact_queries = {q["qid"]: compact_query(q) for q in tqdm(queries, desc="compact queries")}
save_json(compact_queries, f"{BASE}/compact_data/compact_queries.json", indent=2)

# cases
case_jsons = glob.glob(f"{BASE}/refined_cases/*.json")
compact_cases = {}
for fp in tqdm(case_jsons, desc="compact cases"):
    with open(fp, "r", encoding="utf-8") as f:
        obj = json.load(f)
    compact_cases[obj["doc_id"]] = compact_case(obj)
save_json(compact_cases, f"{BASE}/compact_data/compact_cases.json")

# statutes
with open(f"{BASE}/processed_statutes.json", "r", encoding="utf-8") as f:
    statutes = json.load(f)
compact_stats = {s["sid"]: compact_statute(s) for s in tqdm(statutes, desc="compact statutes")}
save_json(compact_stats, f"{BASE}/compact_data/compact_statutes.json")

print(f"  compact_queries={len(compact_queries)}  compact_cases={len(compact_cases)}  compact_stats={len(compact_stats)}")


# ============================================================
# 7. BM25 HARD NEGATIVE MINING  (v3 triplets)
# ============================================================
print("\n[STAGE 7] BM25 hard negative mining...")

query_map = load_json(f"{BASE}/compact_data/compact_queries.json")
case_map  = load_json(f"{BASE}/compact_data/compact_cases.json")
stat_map  = load_json(f"{BASE}/compact_data/compact_statutes.json")
case_pos  = parse_rel(CASE_REL)
stat_pos  = parse_rel(STAT_REL)

def tokenize_bm25(text):
    return re.findall(r"\b[a-zA-Z_]{2,}\b", text.lower())

def build_bm25_triplets(name, doc_map, pos_map, topk=150):
    doc_ids = list(doc_map.keys())
    corpus  = [tokenize_bm25(doc_map[d]) for d in tqdm(doc_ids, desc=f"{name} tokenize")]
    bm25    = BM25Okapi(corpus)
    triplets = []; pairs = []
    for qid in tqdm(pos_map, desc=f"mine {name}"):
        if qid not in query_map: continue
        qtxt  = query_map[qid]
        qtok  = tokenize_bm25(qtxt)
        pos   = [x for x in pos_map[qid] if x in doc_map]
        if not pos: continue
        scores      = bm25.get_scores(qtok)
        ranked_idx  = np.argsort(scores)[::-1][:topk]
        pos_set     = set(pos)
        hard_negs   = [doc_ids[i] for i in ranked_idx if doc_ids[i] not in pos_set and len(doc_map[doc_ids[i]]) >= 80][:25]
        if not hard_negs: continue
        for pid in pos:
            pairs.append({"qid":qid,"doc_id":pid,"query":qtxt,"doc":doc_map[pid],"label":1})
        for nid in hard_negs:
            pairs.append({"qid":qid,"doc_id":nid,"query":qtxt,"doc":doc_map[nid],"label":0})
        for pid in pos:
            for nid in hard_negs[:12]:
                triplets.append({"qid":qid,"query":qtxt,"pos_id":pid,"positive":doc_map[pid],"neg_id":nid,"negative":doc_map[nid]})
    print(f"{name}: {len(triplets)} triplets, {len(pairs)} pairs")
    return triplets, pairs

case_triplets_v3, case_pairs_v3 = build_bm25_triplets("CASE", case_map, case_pos)
stat_triplets_v3, stat_pairs_v3 = build_bm25_triplets("STATUTE", stat_map, stat_pos)

for name, obj in {
    "case_triplets_v3.json": case_triplets_v3,
    "case_pairs_v3.json":    case_pairs_v3,
    "stat_triplets_v3.json": stat_triplets_v3,
    "stat_pairs_v3.json":    stat_pairs_v3,
}.items():
    save_json(obj, f"{BASE}/training_data_v3/{name}")

print("  BM25 mining done.")


# ============================================================
# 8. BI-ENCODER V1  (InLegalBERT, MNRL, bs=16)
# ============================================================
print("\n[STAGE 8] Training bi_encoder_v1 ...")

def make_examples(triplets_list, max_len=3000):
    seen = set(); examples = []
    for x in triplets_list:
        key = (x["query"][:200], x["positive"][:200])
        if key in seen: continue
        seen.add(key)
        examples.append(InputExample(texts=[x["query"][:max_len], x["positive"][:max_len]]))
    return examples

examples_v1 = make_examples(case_triplets_v3 + stat_triplets_v3)
random.shuffle(examples_v1)
print(f"  Examples: {len(examples_v1)}")

model = SentenceTransformer("law-ai/InLegalBERT", device=DEVICE)
model.max_seq_length = 512

loader_v1 = DataLoader(examples_v1, batch_size=16, shuffle=True, drop_last=True, pin_memory=True)
loss_v1   = losses.MultipleNegativesRankingLoss(model)
warmup_v1 = int(len(loader_v1) * 5 * 0.1)

model.fit(
    train_objectives=[(loader_v1, loss_v1)],
    epochs=5,
    warmup_steps=warmup_v1,
    optimizer_params={"lr": 2e-5},
    output_path=f"{BASE}/models/bi_encoder_v1",
    use_amp=True,
    checkpoint_path=f"{BASE}/models/bi_encoder_v1/ckpt",
    checkpoint_save_steps=200,
    show_progress_bar=True,
)
print("  Saved bi_encoder_v1")
del model; gc.collect(); torch.cuda.empty_cache()


# ============================================================
# 9. BI-ENCODER V2  (full dataset, bs=32, 5 epochs)
# ============================================================
print("\n[STAGE 9] Training bi_encoder_v2 ...")

examples_v2 = []
for x in case_triplets_v3 + stat_triplets_v3:
    examples_v2.append(InputExample(texts=[x["query"][:2500], x["positive"][:2500]]))
random.shuffle(examples_v2)
print(f"  Examples: {len(examples_v2)}")

model = SentenceTransformer("law-ai/InLegalBERT", device=DEVICE)
model.max_seq_length = 512

loader_v2 = DataLoader(examples_v2, batch_size=32, shuffle=True, drop_last=True,
                        pin_memory=True, num_workers=4)
loss_v2   = losses.MultipleNegativesRankingLoss(model)
warmup_v2 = int(len(loader_v2) * 5 * 0.1)

model.fit(
    train_objectives=[(loader_v2, loss_v2)],
    epochs=5,
    warmup_steps=warmup_v2,
    optimizer_params={"lr": 2e-5},
    output_path=f"{BASE}/models/bi_encoder_v2",
    use_amp=True,
    checkpoint_path=f"{BASE}/models/bi_encoder_v2/ckpt",
    checkpoint_save_steps=200,
    show_progress_bar=True,
)
print("  Saved bi_encoder_v2")
del model; gc.collect(); torch.cuda.empty_cache()


# ============================================================
# 10. SEMANTIC HARD NEGATIVE MINING  (using bi_encoder_v2)
# ============================================================
print("\n[STAGE 10] Semantic hard negative mining ...")

model = SentenceTransformer(f"{BASE}/models/bi_encoder_v2", device=DEVICE)
model.max_seq_length = 512

def encode(texts, bs=128):
    return model.encode(
        texts, batch_size=bs, show_progress_bar=True,
        convert_to_numpy=True, normalize_embeddings=True,
    ).astype("float32")

def build_faiss(doc_map, name):
    ids   = list(doc_map.keys())
    texts = [doc_map[x] for x in ids]
    print(f"  Encoding {name}...")
    vecs  = encode(texts)
    idx   = faiss.IndexFlatIP(vecs.shape[1])
    idx.add(vecs)
    return ids, texts, idx

case_ids_s, case_texts_s, case_idx_s = build_faiss(case_map, "CASE")
stat_ids_s, stat_texts_s, stat_idx_s = build_faiss(stat_map, "STATUTE")

qids   = list(query_map.keys())
Q      = encode([query_map[q] for q in qids], 64)
qid2v  = {qid: Q[i] for i, qid in enumerate(qids)}

def semantic_mine(name, gt, ids, texts, faiss_index, top=100, n_neg=15, n_trip=5):
    id2txt = {ids[i]: texts[i] for i in range(len(ids))}
    triplets = []; pairs = []
    for qid in tqdm(gt, desc=name):
        if qid not in qid2v: continue
        qvec = qid2v[qid].reshape(1, -1)
        _, idxs = faiss_index.search(qvec, top)
        positives = gt[qid]
        hard_negs = [ids[i] for i in idxs[0] if ids[i] not in positives][:n_neg]
        if not hard_negs: continue
        qtxt = query_map[qid]
        for pid in positives:
            if pid not in id2txt: continue
            ptxt = id2txt[pid]
            pairs.append({"qid":qid,"doc_id":pid,"query":qtxt,"doc":ptxt,"label":1})
            for nid in hard_negs[:n_trip]:
                ntxt = id2txt[nid]
                pairs.append({"qid":qid,"doc_id":nid,"query":qtxt,"doc":ntxt,"label":0})
                triplets.append({"qid":qid,"query":qtxt,"pos_id":pid,"positive":ptxt,"neg_id":nid,"negative":ntxt})
    print(f"{name}: {len(triplets)} triplets, {len(pairs)} pairs")
    return triplets, pairs

case_sem_trip, case_sem_pairs = semantic_mine("CASE",   case_pos, case_ids_s, case_texts_s, case_idx_s)
stat_sem_trip, stat_sem_pairs = semantic_mine("STATUTE", stat_pos, stat_ids_s, stat_texts_s, stat_idx_s)

save_json(case_sem_trip,  f"{BASE}/semantic_hard_v1/case_triplets_semantic.json")
save_json(case_sem_pairs, f"{BASE}/semantic_hard_v1/case_pairs_semantic.json")
save_json(stat_sem_trip,  f"{BASE}/semantic_hard_v1/stat_triplets_semantic.json")
save_json(stat_sem_pairs, f"{BASE}/semantic_hard_v1/stat_pairs_semantic.json")

del model; gc.collect(); torch.cuda.empty_cache()
print("  Semantic mining done.")


# ============================================================
# 11. BI-ENCODER V3  (A100 optimised — Stage 2 hard-negative FT)
#     CachedMNRL, bs=192, mini_batch=48, 4 epochs, lr=1e-5
# ============================================================
print("\n[STAGE 11] Fine-tuning bi_encoder_v3 on A100 ...")

examples_v3 = []
for x in case_sem_trip + stat_sem_trip:
    examples_v3.append(InputExample(texts=[x["query"][:2200], x["positive"][:2200]]))
random.shuffle(examples_v3)
print(f"  Examples: {len(examples_v3)}")

model = SentenceTransformer(f"{BASE}/models/bi_encoder_v2", device=DEVICE)
model.max_seq_length = 512

BATCH_V3       = 192   # A100 80 GB can handle this with CachedMNRL
MINI_BATCH_V3  = 48    # gradient-cache mini-batch size
EPOCHS_V3      = 4
LR_V3          = 1e-5

loader_v3 = DataLoader(
    examples_v3, batch_size=BATCH_V3, shuffle=True, drop_last=True,
    pin_memory=True, num_workers=8, persistent_workers=True,
)
loss_v3 = losses.CachedMultipleNegativesRankingLoss(model, mini_batch_size=MINI_BATCH_V3)
warmup_v3 = int(len(loader_v3) * EPOCHS_V3 * 0.1)
print(f"  Steps/epoch: {len(loader_v3)}  |  Warmup: {warmup_v3}")

model.fit(
    train_objectives=[(loader_v3, loss_v3)],
    epochs=EPOCHS_V3,
    warmup_steps=warmup_v3,
    optimizer_params={"lr": LR_V3},
    output_path=f"{BASE}/models/bi_encoder_v3_a100",
    use_amp=True,
    checkpoint_path=f"{BASE}/models/bi_encoder_v3_a100/ckpt",
    checkpoint_save_steps=25,
    show_progress_bar=True,
)
print("  Saved bi_encoder_v3_a100")
del model; gc.collect(); torch.cuda.empty_cache()


# ============================================================
# 12. EVALUATION  (bi_encoder_v3_a100)
#     Metrics: Precision@10, Recall@10, F1@10, MAP, MRR, nDCG@10
# ============================================================
print("\n[STAGE 12] Evaluation ...")

def apk(actual, predicted, k=10):
    predicted = predicted[:k]; score = 0.0; hits = 0
    for i, p in enumerate(predicted):
        if p in actual and p not in predicted[:i]:
            hits += 1; score += hits / (i + 1)
    return score / min(len(actual), k) if actual else 0.0

def rr(actual, predicted):
    for i, p in enumerate(predicted):
        if p in actual: return 1 / (i + 1)
    return 0.0

def ndcg_k(actual, predicted, k=10):
    predicted = predicted[:k]
    dcg   = sum(1/np.log2(i+2) for i, p in enumerate(predicted) if p in actual)
    ideal = sum(1/np.log2(i+2) for i in range(min(len(actual), k)))
    return dcg / ideal if ideal else 0.0

model = SentenceTransformer(f"{BASE}/models/bi_encoder_v3_a100", device=DEVICE)
model.max_seq_length = 512

def encode_eval(texts, bs=128):
    return model.encode(
        texts, batch_size=bs, show_progress_bar=True,
        convert_to_numpy=True, normalize_embeddings=True,
    ).astype("float32")

def build_eval_index(doc_map, name):
    ids   = list(doc_map.keys())
    texts = [doc_map[x] for x in ids]
    print(f"  Encoding {name}...")
    vecs  = encode_eval(texts)
    idx   = faiss.IndexFlatIP(vecs.shape[1])
    idx.add(vecs)
    return ids, idx

case_ids_e, case_idx_e = build_eval_index(case_map, "CASE")
stat_ids_e, stat_idx_e = build_eval_index(stat_map, "STATUTE")

qids_e  = list(query_map.keys())
Q_e     = encode_eval([query_map[q] for q in qids_e], 64)
qid2v_e = {qid: Q_e[i] for i, qid in enumerate(qids_e)}

case_gt = parse_rel(CASE_REL)
stat_gt = parse_rel(STAT_REL)

def evaluate(name, gt, ids, faiss_index):
    P, R, F, MAP, MRR, NDCG = [], [], [], [], [], []
    for qid in tqdm(gt, desc=name):
        if qid not in qid2v_e: continue
        qvec = qid2v_e[qid].reshape(1, -1)
        _, idxs = faiss_index.search(qvec, 10)
        preds = [ids[i] for i in idxs[0]]
        truth = gt[qid]
        hit   = len(set(preds) & truth)
        prec  = hit / 10
        rec   = hit / len(truth) if truth else 0
        f1    = 2*prec*rec/(prec+rec) if prec+rec else 0
        P.append(prec); R.append(rec); F.append(f1)
        MAP.append(apk(truth, preds)); MRR.append(rr(truth, preds)); NDCG.append(ndcg_k(truth, preds))
    metrics = {
        "Precision@10": round(float(np.mean(P)),  4),
        "Recall@10":    round(float(np.mean(R)),  4),
        "F1@10":        round(float(np.mean(F)),  4),
        "MAP":          round(float(np.mean(MAP)),4),
        "MRR":          round(float(np.mean(MRR)),4),
        "nDCG@10":      round(float(np.mean(NDCG)),4),
    }
    print(f"\n{name} Results:")
    for k, v in metrics.items():
        print(f"  {k:<15} {v:.4f}")
    return metrics

print("\n=== CASE RETRIEVAL ===")
case_metrics = evaluate("CASE",    case_gt, case_ids_e, case_idx_e)

print("\n=== STATUTE RETRIEVAL ===")
stat_metrics = evaluate("STATUTE", stat_gt, stat_ids_e, stat_idx_e)

final_results = {"CASE": case_metrics, "STATUTE": stat_metrics}
save_json(final_results, f"{BASE}/eval_results_v3_a100.json", indent=2)
print(f"\nResults saved to {BASE}/eval_results_v3_a100.json")
print("\nPIPELINE COMPLETE ✓")

/usr/local/lib/python3.10/dist-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_195939/1052205704.py:49: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, InputExample, losses


GPU : NVIDIA A100-SXM4-80GB MIG 3g.40gb
VRAM: 42.14 GB

[STAGE 3] Preprocessing cases...
  Found 2914 case files


cases: 100%|██████████| 2914/2914 [01:29<00:00, 32.56it/s]


  OK=2914  Failed=0

[STAGE 4] Preprocessing queries...
  Queries: 50

[STAGE 5] Preprocessing statutes...
  Found 197 statute files


statutes: 100%|██████████| 197/197 [00:00<00:00, 1229.12it/s]


...


=== CASE RETRIEVAL ===


CASE: 100%|██████████| 50/50 [00:00<00:00, 3874.14it/s]



CASE Results:
  Precision@10    0.3120
  Recall@10       0.9301
  F1@10           0.4196
  MAP             0.8694
  MRR             0.8922
  nDCG@10         0.9036

=== STATUTE RETRIEVAL ===


STATUTE: 100%|██████████| 50/50 [00:00<00:00, 15407.77it/s]


STATUTE Results:
  Precision@10    0.4300
  Recall@10       0.9687
  F1@10           0.5927
  MAP             0.8880
  MRR             0.9550
  nDCG@10         0.9360

Results saved to /workspace/aila/ailanew/aila/eval_results_v3_a100.json

PIPELINE COMPLETE ✓


In [7]:
"""
============================================================
SUPPLEMENTARY CODE
Loss Curve Plotting + Baseline Comparison Evaluation
AILA Legal Retrieval Project
============================================================
"""

# ── SECTION A: LOSS CURVE TRACKING ──────────────────────────────────────────
# Wrap model.fit() with a custom callback to capture training loss per step.
# Run this instead of the plain model.fit() calls in the main pipeline.

import os
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sentence_transformers import SentenceTransformer, InputExample, losses, evaluation
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from torch.utils.data import DataLoader

BASE = "/workspace/aila/ailanew/aila"


# ── A1. Custom loss logger ───────────────────────────────────────────────────
class LossLogger:
    """Plugs into sentence_transformers' callback_fn to record per-step loss."""
    def __init__(self):
        self.losses = []   # list of (step, loss)
        self._step  = 0

    def __call__(self, score, epoch, steps):
        """Called after every eval; we use this as a proxy checkpoint."""
        self.losses.append({"epoch": epoch, "steps": steps, "score": score})
        self._step = steps

    def record_train_loss(self, loss_value):
        self.losses.append({"step": self._step, "train_loss": float(loss_value)})


# ── A2. Train with loss tracking (monkey-patch approach) ────────────────────
def train_with_loss_tracking(model, loader, loss_fn, epochs, warmup_steps,
                              lr, save_dir, tag="model"):
    """
    sentence-transformers >=3.x exposes per-step loss in the progress bar.
    We capture it by subclassing the loss and logging inside forward().
    """
    tracked_losses = []

    # Wrap CachedMNRL to intercept loss values
    original_forward = loss_fn.forward

    step_counter = [0]

    def logging_forward(*args, **kwargs):
        result = original_forward(*args, **kwargs)
        # result is the loss tensor
        step_counter[0] += 1
        tracked_losses.append({
            "step": step_counter[0],
            "loss": float(result.detach().cpu().item()) if hasattr(result, "item") else float(result)
        })
        return result

    loss_fn.forward = logging_forward

    model.fit(
        train_objectives=[(loader, loss_fn)],
        epochs=epochs,
        warmup_steps=warmup_steps,
        optimizer_params={"lr": lr},
        output_path=save_dir,
        use_amp=True,
        show_progress_bar=True,
    )

    # Save loss log
    log_path = f"{save_dir}/loss_log.json"
    with open(log_path, "w") as f:
        json.dump(tracked_losses, f, indent=2)

    print(f"Loss log saved to {log_path}")
    return tracked_losses


# ── A3. Plot loss curves ─────────────────────────────────────────────────────
def plot_loss_curves(logs_dict, save_path="loss_curves.png"):
    """
    logs_dict: {"v1": [...loss dicts...], "v2": [...], "v3": [...]}
    Each list contains {"step": int, "loss": float}
    """
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle("Training Loss Curves — AILA Bi-Encoder Stages",
                 fontsize=14, fontweight="bold", y=1.02)

    colours = {"v1": "#2196F3", "v2": "#4CAF50", "v3": "#F44336"}
    labels  = {
        "v1": "Bi-Encoder v1\n(MNRL, bs=16, 5 epochs)",
        "v2": "Bi-Encoder v2\n(MNRL, bs=32, 5 epochs)",
        "v3": "Bi-Encoder v3 — A100\n(CachedMNRL, bs=192, 4 epochs)"
    }

    for ax, (tag, log) in zip(axes, logs_dict.items()):
        steps  = [x["step"] for x in log if "loss" in x]
        losses = [x["loss"] for x in log if "loss" in x]

        # Smooth with a rolling window
        win    = max(1, len(steps) // 30)
        smooth = np.convolve(losses, np.ones(win)/win, mode="valid")
        s_steps = steps[win-1:]

        ax.plot(steps, losses, alpha=0.25, color=colours[tag], linewidth=0.8)
        ax.plot(s_steps, smooth, color=colours[tag], linewidth=2.2, label="Smoothed")

        ax.set_title(labels[tag], fontsize=11, pad=8)
        ax.set_xlabel("Training Step", fontsize=10)
        ax.set_ylabel("CachedMNRL Loss" if tag == "v3" else "MNRL Loss", fontsize=10)
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))
        ax.spines[["top","right"]].set_visible(False)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Loss curve saved to {save_path}")


# ── A4. Load saved logs and plot (run after training) ───────────────────────
def generate_loss_plot():
    logs = {}
    for tag in ["bi_encoder_v1", "bi_encoder_v2", "bi_encoder_v3_a100"]:
        path = f"{BASE}/models/{tag}/loss_log.json"
        short = {"bi_encoder_v1":"v1","bi_encoder_v2":"v2","bi_encoder_v3_a100":"v3"}[tag]
        if os.path.exists(path):
            with open(path) as f:
                logs[short] = json.load(f)
        else:
            print(f"Warning: {path} not found. Run training with loss tracking first.")

    if logs:
        plot_loss_curves(logs, save_path=f"{BASE}/loss_curves_all_stages.png")


# ═══════════════════════════════════════════════════════════════════════════
# SECTION B: BASELINE COMPARISON EVALUATION
# Evaluates BM25, v1, v2, v3 side-by-side and plots a comparison bar chart
# ═══════════════════════════════════════════════════════════════════════════

import faiss
import re
from collections import defaultdict
from tqdm.auto import tqdm
from rank_bm25 import BM25Okapi


# ── B1. Metric functions ─────────────────────────────────────────────────────
def apk(actual, predicted, k=10):
    predicted = predicted[:k]; score = 0.0; hits = 0
    for i, p in enumerate(predicted):
        if p in actual and p not in predicted[:i]:
            hits += 1; score += hits / (i + 1)
    return score / min(len(actual), k) if actual else 0.0

def rr(actual, predicted):
    for i, p in enumerate(predicted):
        if p in actual: return 1 / (i + 1)
    return 0.0

def ndcg_k(actual, predicted, k=10):
    predicted = predicted[:k]
    dcg   = sum(1/np.log2(i+2) for i, p in enumerate(predicted) if p in actual)
    ideal = sum(1/np.log2(i+2) for i in range(min(len(actual), k)))
    return dcg / ideal if ideal else 0.0

def compute_metrics(gt, preds_dict):
    """preds_dict: {qid: [ranked doc_ids]}"""
    P, R, F, MAP, MRR, NDCG = [], [], [], [], [], []
    for qid, truth in gt.items():
        preds = preds_dict.get(qid, [])
        hit   = len(set(preds[:10]) & truth)
        prec  = hit / 10
        rec   = hit / len(truth) if truth else 0
        f1    = 2*prec*rec/(prec+rec) if prec+rec else 0
        P.append(prec); R.append(rec); F.append(f1)
        MAP.append(apk(truth, preds)); MRR.append(rr(truth, preds)); NDCG.append(ndcg_k(truth, preds))
    return {
        "Precision@10": round(float(np.mean(P)),  4),
        "Recall@10":    round(float(np.mean(R)),  4),
        "F1@10":        round(float(np.mean(F)),  4),
        "MAP":          round(float(np.mean(MAP)),4),
        "MRR":          round(float(np.mean(MRR)),4),
        "nDCG@10":      round(float(np.mean(NDCG)),4),
    }


# ── B2. BM25 retrieval ───────────────────────────────────────────────────────
def tokenize(text):
    return re.findall(r"\b[a-zA-Z_]{2,}\b", text.lower())

def run_bm25(query_map, doc_map, gt, name="BM25"):
    print(f"\nBuilding {name} index...")
    doc_ids  = list(doc_map.keys())
    corpus   = [tokenize(doc_map[d]) for d in tqdm(doc_ids, desc="tokenize")]
    bm25     = BM25Okapi(corpus)
    preds    = {}
    for qid in tqdm(gt, desc=f"{name} retrieval"):
        if qid not in query_map: continue
        scores     = bm25.get_scores(tokenize(query_map[qid]))
        ranked     = np.argsort(scores)[::-1][:10]
        preds[qid] = [doc_ids[i] for i in ranked]
    return compute_metrics(gt, preds)


# ── B3. Dense retrieval (bi-encoder) ────────────────────────────────────────
def run_dense(model_dir, query_map, doc_map, gt, name, device="cuda"):
    print(f"\nLoading {name} from {model_dir}...")
    model = SentenceTransformer(model_dir, device=device)
    model.max_seq_length = 512

    def encode(texts, bs=128):
        return model.encode(texts, batch_size=bs, show_progress_bar=True,
                            convert_to_numpy=True, normalize_embeddings=True).astype("float32")

    doc_ids   = list(doc_map.keys())
    doc_vecs  = encode([doc_map[d] for d in doc_ids])
    query_ids = list(query_map.keys())
    query_vecs= encode([query_map[q] for q in query_ids], bs=64)

    idx = faiss.IndexFlatIP(doc_vecs.shape[1])
    idx.add(doc_vecs)

    qid2v = {qid: query_vecs[i] for i, qid in enumerate(query_ids)}

    preds = {}
    for qid in tqdm(gt, desc=f"{name} retrieval"):
        if qid not in qid2v: continue
        _, idxs    = idx.search(qid2v[qid].reshape(1, -1), 10)
        preds[qid] = [doc_ids[i] for i in idxs[0]]

    del model
    import gc, torch
    gc.collect(); torch.cuda.empty_cache()
    return compute_metrics(gt, preds)


# ── B4. Run all comparisons ──────────────────────────────────────────────────
def run_full_comparison():
    # Load data
    query_map = json.load(open(f"{BASE}/compact_data/compact_queries.json"))
    case_map  = json.load(open(f"{BASE}/compact_data/compact_cases.json"))
    stat_map  = json.load(open(f"{BASE}/compact_data/compact_statutes.json"))

    def parse_rel(path):
        gt = defaultdict(set)
        with open(path) as f:
            for line in f:
                p = line.strip().split()
                if len(p) == 4 and int(p[3]) == 1:
                    gt[p[0]].add(p[2])
        return gt

    case_gt = parse_rel(f"{BASE}/relevance_judgments_priorcases.txt")
    stat_gt = parse_rel(f"{BASE}/relevance_judgments_statutes.txt")

    models = {
        "BM25":            None,
        "bi_encoder_v1":   f"{BASE}/models/bi_encoder_v1",
        "bi_encoder_v2":   f"{BASE}/models/bi_encoder_v2",
        "bi_encoder_v3":   f"{BASE}/models/bi_encoder_v3_a100",
    }

    case_results = {}
    stat_results = {}

    for name, model_dir in models.items():
        if model_dir is None:  # BM25
            case_results[name] = run_bm25(query_map, case_map, case_gt, name)
            stat_results[name] = run_bm25(query_map, stat_map, stat_gt, name)
        else:
            case_results[name] = run_dense(model_dir, query_map, case_map, case_gt, name)
            stat_results[name] = run_dense(model_dir, query_map, stat_map, stat_gt, name)

        print(f"\n{name} | CASE:    {case_results[name]}")
        print(f"{name} | STATUTE: {stat_results[name]}")

    # Save
    json.dump({"case": case_results, "statute": stat_results},
              open(f"{BASE}/full_comparison.json", "w"), indent=2)

    # Plot
    plot_comparison(case_results, stat_results, f"{BASE}/comparison_charts.png")
    return case_results, stat_results


# ── B5. Bar chart comparison ─────────────────────────────────────────────────
def plot_comparison(case_results, stat_results, save_path="comparison_charts.png"):
    metrics = ["Precision@10", "Recall@10", "F1@10", "MAP", "MRR", "nDCG@10"]
    model_names = list(case_results.keys())
    n_models = len(model_names)
    x = np.arange(len(metrics))
    width = 0.18
    colours = ["#90A4AE", "#42A5F5", "#66BB6A", "#EF5350"]

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

    for i, (name, colour) in enumerate(zip(model_names, colours)):
        vals_case = [case_results[name][m] for m in metrics]
        vals_stat = [stat_results[name][m] for m in metrics]
        ax1.bar(x + i*width - (n_models-1)*width/2, vals_case, width, label=name, color=colour, edgecolor="white")
        ax2.bar(x + i*width - (n_models-1)*width/2, vals_stat, width, label=name, color=colour, edgecolor="white")

    for ax, title in [(ax1, "Case Retrieval"), (ax2, "Statute Retrieval")]:
        ax.set_title(title, fontsize=13, fontweight="bold", pad=10)
        ax.set_xticks(x); ax.set_xticklabels(metrics, fontsize=10)
        ax.set_ylim(0, 1.05)
        ax.set_ylabel("Score", fontsize=11)
        ax.legend(fontsize=9, loc="upper left")
        ax.grid(axis="y", alpha=0.3)
        ax.spines[["top","right"]].set_visible(False)
        for bar_group in ax.containers:
            ax.bar_label(bar_group, fmt="%.3f", fontsize=7, padding=2, rotation=90)

    plt.suptitle("AILA Retrieval — Model Comparison Across All Metrics",
                 fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Comparison chart saved to {save_path}")


# ── B6. MRR / nDCG focused comparison (for report) ──────────────────────────
def plot_ranking_metrics(case_results, stat_results, save_path="ranking_metrics.png"):
    """Clean two-metric chart focusing on MAP, MRR, nDCG@10."""
    metrics   = ["MAP", "MRR", "nDCG@10"]
    model_names = list(case_results.keys())

    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)
    colours   = ["#90A4AE", "#42A5F5", "#66BB6A", "#EF5350"]
    x = np.arange(len(metrics)); width = 0.18

    for ax, (results, title) in zip(axes, [
        (case_results,  "Case Retrieval — Ranking Metrics"),
        (stat_results,  "Statute Retrieval — Ranking Metrics"),
    ]):
        for i, (name, colour) in enumerate(zip(model_names, colours)):
            vals = [results[name][m] for m in metrics]
            bars = ax.bar(x + i*width - (len(model_names)-1)*width/2, vals,
                          width, label=name, color=colour, edgecolor="white", linewidth=0.5)
            ax.bar_label(bars, fmt="%.3f", fontsize=8, padding=3)

        ax.set_title(title, fontsize=12, fontweight="bold")
        ax.set_xticks(x); ax.set_xticklabels(metrics, fontsize=11)
        ax.set_ylim(0, 1.10)
        ax.set_ylabel("Score", fontsize=11)
        ax.legend(fontsize=9)
        ax.grid(axis="y", alpha=0.25)
        ax.spines[["top","right"]].set_visible(False)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Ranking metrics chart saved to {save_path}")


# ══════════════════════════════════════════════════════════════════════
# MAIN — run whichever sections you need
# ══════════════════════════════════════════════════════════════════════
# if __name__ == "__main__":
#     import argparse
#     parser = argparse.ArgumentParser()
#     parser.add_argument("--loss",    action="store_true", help="Plot loss curves from saved logs")
#     parser.add_argument("--compare", action="store_true", help="Run full baseline comparison + charts")
#     args = parser.parse_args()

#     if args.loss:
#         generate_loss_plot()

#     if args.compare:
#         case_res, stat_res = run_full_comparison()
#         plot_ranking_metrics(case_res, stat_res, f"{BASE}/ranking_metrics.png")

#     if not args.loss and not args.compare:
#         print("Usage: python supplementary_code.py --loss --compare")
#         print("  --loss    : generate loss curve plots from training logs")
#         print("  --compare : run all 4 models (BM25, v1, v2, v3) and plot comparison")
# ── Notebook-compatible runner ──────────────────────────────
# Instead of argparse, just set these flags directly:

RUN_LOSS_CURVES   = True   # set False if loss logs don't exist yet
RUN_COMPARISON    = True   # set False to skip re-running all models

if RUN_LOSS_CURVES:
    generate_loss_plot()

if RUN_COMPARISON:
    case_res, stat_res = run_full_comparison()
    plot_ranking_metrics(case_res, stat_res, f"{BASE}/ranking_metrics.png")


Building BM25 index...


BM25 retrieval: 100%|██████████| 50/50 [00:05<00:00,  8.94it/s]



Building BM25 index...


BM25 retrieval: 100%|██████████| 50/50 [00:00<00:00, 140.87it/s]



BM25 | CASE:    {'Precision@10': 0.022, 'Recall@10': 0.0787, 'F1@10': 0.029, 'MAP': 0.0382, 'MRR': 0.0886, 'nDCG@10': 0.0592}
BM25 | STATUTE: {'Precision@10': 0.044, 'Recall@10': 0.1013, 'F1@10': 0.061, 'MAP': 0.0428, 'MRR': 0.1608, 'nDCG@10': 0.0874}

Loading bi_encoder_v1 from /workspace/aila/ailanew/aila/models/bi_encoder_v1...


...


bi_encoder_v2 | CASE:    {'Precision@10': 0.308, 'Recall@10': 0.9243, 'F1@10': 0.4149, 'MAP': 0.8603, 'MRR': 0.8719, 'nDCG@10': 0.8943}
bi_encoder_v2 | STATUTE: {'Precision@10': 0.428, 'Recall@10': 0.9647, 'F1@10': 0.59, 'MAP': 0.8801, 'MRR': 0.9467, 'nDCG@10': 0.93}

Loading bi_encoder_v3 from /workspace/aila/ailanew/aila/models/bi_encoder_v3_a100...


bi_encoder_v3 retrieval: 100%|██████████| 50/50 [00:00<00:00, 4322.24it/s]



Loading bi_encoder_v3 from /workspace/aila/ailanew/aila/models/bi_encoder_v3_a100...


bi_encoder_v3 retrieval: 100%|██████████| 50/50 [00:00<00:00, 33961.98it/s]



bi_encoder_v3 | CASE:    {'Precision@10': 0.312, 'Recall@10': 0.9301, 'F1@10': 0.4196, 'MAP': 0.8694, 'MRR': 0.8922, 'nDCG@10': 0.9036}
bi_encoder_v3 | STATUTE: {'Precision@10': 0.43, 'Recall@10': 0.9687, 'F1@10': 0.5927, 'MAP': 0.888, 'MRR': 0.955, 'nDCG@10': 0.936}
Comparison chart saved to /workspace/aila/ailanew/aila/comparison_charts.png
Ranking metrics chart saved to /workspace/aila/ailanew/aila/ranking_metrics.png


SystemExit: 2